In [52]:
# Analiza kontekstu otoczenia twarzy w stosunku do reszty obrazu
# Wczytanie przykładowych danych
import cv2
import cv2.data
import numpy as np
import matplotlib.pyplot as plt

# Wczytanie obrazu
img = cv2.imread('../../../Dane/Lenna.png', cv2.IMREAD_COLOR)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

plt.imshow(img)
plt.axis('off')
plt.show()

In [53]:
# Wykrycie twarzy na obrazie
face_classifier = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

faces = face_classifier.detectMultiScale(img, 1.3, 5)
img_copy = img.copy()

for (x, y, w, h) in faces:
    cv2.rectangle(img_copy, (x, y), (x+w, y+h), (255, 0, 0), 2)

plt.imshow(img_copy)
plt.axis('off')
plt.show()

In [54]:
# Obliczenie kontekstu otoczenia twarzy
def get_face_context(img, face_coordinates):
    # 1. Obliczenie proporcji twarzy w stosunku do całego obrazu
    # 2. Obliczenie współrzędnych punktów w bezpośrednim otoczeniu twarzy
    # 3. Obliczenie współrzędnych punktów w dalszym otoczeniu twarzy
    # 4. Zwrócenie współrzędnych z funkcji
    x, y, w, h = face_coordinates
    img_h, img_w, _ = img.shape
    
    # Obliczenie proporcji twarzy w stosunku do całego obrazu
    face_proportion = (w * h) / (img_h * img_w)
    
    # Uznajemy, że bezpośrednie otoczenie twarzy to 40% wysokości i szerokości twarzy
    # [PARAMETR]
    context_proportion = 0.40
    
    # Obliczenie współrzędnych punktów w bezpośrednim otoczeniu twarzy
    rectangle_point = (x - (context_proportion * w), y - (context_proportion * h))
    rectangle_width = w + 2 * (context_proportion * w)
    rectangle_height = h + 2 * (context_proportion * h)
    
    # Obliczenie współrzędnych punktów w dalszym otoczeniu twarzy
    rectangle_point_far = (x - (2 * context_proportion * w), y - (2 * context_proportion * h))
    rectangle_width_far = w + 4 * (context_proportion * w)
    rectangle_height_far = h + 4 * (context_proportion * h)
    
    # Sprawdzenie, czy punkty nie wychodzą poza obraz
    if rectangle_point[0] < 0:
        rectangle_point = (0, rectangle_point[1])
    if rectangle_point[1] < 0:
        rectangle_point = (rectangle_point[0], 0)
    if rectangle_point[0] + rectangle_width > img_w:
        rectangle_width = img_w - rectangle_point[0]
    if rectangle_point[1] + rectangle_height > img_h:
        rectangle_height = img_h - rectangle_point[1]
        
    if rectangle_point_far[0] < 0:
        rectangle_point_far = (0, rectangle_point_far[1])
    if rectangle_point_far[1] < 0:
        rectangle_point_far = (rectangle_point_far[0], 0)
    if rectangle_point_far[0] + rectangle_width_far > img_w:
        rectangle_width_far = img_w - rectangle_point_far[0]
    if rectangle_point_far[1] + rectangle_height_far > img_h:
        rectangle_height_far = img_h - rectangle_point_far[1]
    
    # Zwrócenie wartości
    return {
        'face_proportion': face_proportion,
        'rectangle_point': rectangle_point,
        'rectangle_width': rectangle_width,
        'rectangle_height': rectangle_height,
        'rectangle_point_far': rectangle_point_far,
        'rectangle_width_far': rectangle_width_far,
        'rectangle_height_far': rectangle_height_far
    }

In [55]:
# Testowanie funkcji
face_context = get_face_context(img, faces[0])
print(face_context)

In [56]:
# Wizualizacja kontekstu otoczenia twarzy
img_copy = img.copy()
x, y, w, h = faces[0]

# Twarz
cv2.rectangle(img_copy, (x, y), (x+w, y+h), (255, 0, 0), 2)

# Bezpośrednie otoczenie twarzy
rectangle_point = (int(face_context['rectangle_point'][0]), int(face_context['rectangle_point'][1]))
rectangle_width = int(face_context['rectangle_width'])
rectangle_height = int(face_context['rectangle_height'])
cv2.rectangle(img_copy, rectangle_point, (rectangle_point[0] + rectangle_width, rectangle_point[1] + rectangle_height), (0, 255, 0), 2)

# Dalsze otoczenie twarzy
rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
rectangle_width_far = int(face_context['rectangle_width_far'])
rectangle_height_far = int(face_context['rectangle_height_far'])
cv2.rectangle(img_copy, rectangle_point_far, (rectangle_point_far[0] + rectangle_width_far, rectangle_point_far[1] + rectangle_height_far), (0, 0, 255), 2)

plt.imshow(img_copy)
plt.axis('off')
plt.show()

In [57]:
# Skopiowanie obszarów obrazu
img_copy = img.copy()
face_area = img_copy[y:y+h, x:x+w]
context_area = img_copy[rectangle_point[1]:rectangle_point[1] + rectangle_height, rectangle_point[0]:rectangle_point[0] + rectangle_width]
context_area_far = img_copy[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]

plt.figure(figsize=(10, 10))
plt.subplot(1, 4, 1), plt.imshow(img_copy, cmap='gray'), plt.title("Original Image")
plt.subplot(1, 4, 2), plt.imshow(face_area, cmap='gray'), plt.title("Face Area")
plt.subplot(1, 4, 3), plt.imshow(context_area, cmap='gray'), plt.title("Context Area")
plt.subplot(1, 4, 4), plt.imshow(context_area_far, cmap='gray'), plt.title("Context Area Far")
plt.show()

In [58]:
# Wykrywanie krawędzi (Sobel)
# Sprawdzany będzie tylko region wokół twarzy o podanej szerokości (w) i wysokości (h)
def calculate_edge_intensity(img_sobel, face_topleft, face_botright):
    # Uzyskanie regionu bez samej twarzy
    x1, y1 = face_topleft
    x2, y2 = face_botright
    # 1. [:y1, :] - górna część obrazu
    # 2. [y2:, :] - dolna część obrazu
    # 3. [y1:y2, :x1] - lewa część obrazu bez powtórzeń
    # 4. [y1:y2, x2:] - prawa część obrazu bez powtórzeń
    top = img_sobel[:y1, :]
    bottom = img_sobel[y2:, :]
    left = img_sobel[y1:y2, :x1]
    right = img_sobel[y1:y2, x2:]
    
    # Obliczenie intensywności krawędzi dla wszystkich regionów
    top_sum = np.sum(top)
    bottom_sum = np.sum(bottom)
    left_sum = np.sum(left)
    right_sum = np.sum(right)
    
    top_count = np.count_nonzero(top)
    bottom_count = np.count_nonzero(bottom)
    left_count = np.count_nonzero(left)
    right_count = np.count_nonzero(right)
    
    # Obliczenie średniej intensywności krawędzi
    avg_intensity = (top_sum + bottom_sum + left_sum + right_sum) / (top_count + bottom_count + left_count + right_count)
    
    return avg_intensity

In [59]:
# Sprawdzenie krawędzi
# + dodanie thresholdingu dla każdego wywołania !!!
face_area_gray = cv2.cvtColor(face_area, cv2.COLOR_RGB2GRAY)
context_area_gray = cv2.cvtColor(context_area, cv2.COLOR_RGB2GRAY)
context_area_far_gray = cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY)

context_area_edges_x = cv2.Sobel(face_area_gray, cv2.CV_64F, 1, 0, ksize=3)
context_area_edges_x = cv2.convertScaleAbs(context_area_edges_x)
context_area_edges_y = cv2.Sobel(face_area_gray, cv2.CV_64F, 0, 1, ksize=3)
context_area_edges_y = cv2.convertScaleAbs(context_area_edges_y)

context_area_edges_x_far = cv2.Sobel(context_area_far_gray, cv2.CV_64F, 1, 0, ksize=3)
context_area_edges_x_far = cv2.convertScaleAbs(context_area_edges_x_far)
context_area_edges_y_far = cv2.Sobel(context_area_far_gray, cv2.CV_64F, 0, 1, ksize=3)
context_area_edges_y_far = cv2.convertScaleAbs(context_area_edges_y_far)

plt.figure(figsize=(10, 10))
plt.subplot(1, 5, 1), plt.imshow(img_copy, cmap='gray'), plt.title("Original Image")
plt.subplot(1, 5, 2), plt.imshow(context_area_edges_x, cmap='gray'), plt.title("Sobel X close")
plt.subplot(1, 5, 3), plt.imshow(context_area_edges_y, cmap='gray'), plt.title("Sobel Y close")
plt.subplot(1, 5, 4), plt.imshow(context_area_edges_x_far, cmap='gray'), plt.title("Sobel X far")
plt.subplot(1, 5, 5), plt.imshow(context_area_edges_y_far, cmap='gray'), plt.title("Sobel Y far")
plt.show()

In [60]:
# Obliczenie intensywności dla krawędzi
# Najpierw trzeba uzyskać punkty twarzy na wyciętych fragmentach

# Dla bliższego otoczenia:
# gap_w = context_area_width - face_area_width / 2
# gap_h = context_area_height - face_area_height / 2
# face_topleft = (gap_w, gap_h)
# face_botright = (gap_w + face_area_width, gap_h + face_area_height)
gap_w = (context_area_gray.shape[1] - face_area_gray.shape[1]) // 2
gap_h = (context_area_gray.shape[0] - face_area_gray.shape[0]) // 2
face_topleft = (gap_w, gap_h)
face_botright = (gap_w + face_area_gray.shape[1], gap_h + face_area_gray.shape[0])
context_intensity_x = calculate_edge_intensity(context_area_edges_x, face_topleft, face_botright)
context_intensity_y = calculate_edge_intensity(context_area_edges_y, face_topleft, face_botright)

print("Dla bliższego otoczenia:")
print("Intensywność krawędzi w osi X:", context_intensity_x)
print("Intensywność krawędzi w osi Y:", context_intensity_y)

# Dla dalszego otoczenia:
gap_w = (context_area_far_gray.shape[1] - face_area_gray.shape[1]) // 2
gap_h = (context_area_far_gray.shape[0] - face_area_gray.shape[0]) // 2
face_topleft = (gap_w, gap_h)
face_botright = (gap_w + face_area_gray.shape[1], gap_h + face_area_gray.shape[0])
context_far_intensity_x = calculate_edge_intensity(context_area_edges_x_far, face_topleft, face_botright)
context_far_intensity_y = calculate_edge_intensity(context_area_edges_y_far, face_topleft, face_botright)

print("Dla dalszego otoczenia:")
print("Intensywność krawędzi w osi X:", context_far_intensity_x)
print("Intensywność krawędzi w osi Y:", context_far_intensity_y)

In [61]:
# Porównanie intensywności obrazów
def calculate_image_intensity(img_gray):
    return np.mean(img_gray)


face_intensity = calculate_image_intensity(face_area_gray)
context_intensity = calculate_image_intensity(context_area_gray)
context_intensity_far = calculate_image_intensity(context_area_far_gray)

print("Intensywność samej twarzy:", face_intensity)
print("Intensywność obszaru bliższego twarzy:", context_intensity)
print("Intensywność obszaru dalszego twarzy:", context_intensity_far)

# Logika porównania
# [PARAMETR]
threshold = 0.1

value = abs(context_intensity_far - context_intensity) / face_intensity
print("Wartość intensywności:", value)

if value > threshold:
    print("Wartość przekroczyła próg!")

In [62]:
# Testy - sprawdzenie intensywności dla różnych obrazów
import os

image_path = "../../../Dane/Sample/zd"

# Wczytanie obrazów
files = [f for f in os.listdir(image_path) if f.endswith(".jpg") or f.endswith(".png") or f.endswith(".webp")] # Tylko pliki JPG i PNG i WEBP

images = []
for file in files:
    img = cv2.imread(str(os.path.join(image_path, file)), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append(img)
    
i = 1

# Przeprowadzenie analizy intensywności dla każdego obrazu
for img in images:
    faces = face_classifier.detectMultiScale(img, 1.3, 5)
    
    if faces is None or len(faces) == 0:
        continue
    
    face = faces[0]
    face_context = get_face_context(img, face)
    x, y, w, h = face

    # Bezpośrednie otoczenie twarzy
    rectangle_point = (int(face_context['rectangle_point'][0]), int(face_context['rectangle_point'][1]))
    rectangle_width = int(face_context['rectangle_width'])
    rectangle_height = int(face_context['rectangle_height'])
    
    # Dalsze otoczenie twarzy
    rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
    rectangle_width_far = int(face_context['rectangle_width_far'])
    rectangle_height_far = int(face_context['rectangle_height_far'])
    
    # Uzyskanie fragmentów obrazu
    face_area = img[y:y+h, x:x+w]
    context_area = img[rectangle_point[1]:rectangle_point[1] + rectangle_height, rectangle_point[0]:rectangle_point[0] + rectangle_width]
    context_area_far = img[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]
    
    # Obliczenie intensywności
    face_intensity = calculate_image_intensity(cv2.cvtColor(face_area, cv2.COLOR_RGB2GRAY))
    context_intensity = calculate_image_intensity(cv2.cvtColor(context_area, cv2.COLOR_RGB2GRAY))
    context_intensity_far = calculate_image_intensity(cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY))
    
    print("Intensywność samej twarzy:", face_intensity)
    print("Intensywność obszaru bliższego twarzy:", context_intensity)
    print("Intensywność obszaru dalszego twarzy:", context_intensity_far)
    
    # Logika porównania
    # [PARAMETR]
    threshold = 0.15
    
    value = abs(context_intensity_far - context_intensity) / face_intensity
    print("Wartość intensywności:", value)
    
    if value > threshold:
        print("Wartość przekroczyła próg!")
        plt.subplot(1, 5, i)
        plt.imshow(context_area_far)
        plt.title("Dalszy kontekst obrazu")
        plt.axis('off')
        i += 1
        
plt.show()

In [63]:
# Do pracy:
# 1. Sprawdzenie wartości intensywności dla Lenny + obliczenia
# 2. Sprawdzenie powyższych podstawionych zdjęć (wygląd obrazu + wartość intensywności)
# 3. Sprawdzenie autentycznego zdjęcia (wygląd obrazu + wartość intensywności)

image1 = cv2.imread('../../../Dane/Sample/zd/5.webp', cv2.IMREAD_COLOR)
image1 = cv2.cvtColor(image1, cv2.COLOR_BGR2RGB)
faces = face_classifier.detectMultiScale(image1, 1.3, 5)
face = faces[0]
face_context = get_face_context(image1, face)
x, y, w, h = face

# Bezpośrednie otoczenie twarzy
rectangle_point = (int(face_context['rectangle_point'][0]), int(face_context['rectangle_point'][1]))
rectangle_width = int(face_context['rectangle_width'])
rectangle_height = int(face_context['rectangle_height'])

# Dalsze otoczenie twarzy
rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
rectangle_width_far = int(face_context['rectangle_width_far'])
rectangle_height_far = int(face_context['rectangle_height_far'])

# Uzyskanie fragmentów obrazu
face_area = image1[y:y+h, x:x+w]
context_area = image1[rectangle_point[1]:rectangle_point[1] + rectangle_height, rectangle_point[0]:rectangle_point[0] + rectangle_width]
context_area_far = image1[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]

# Obliczenie intensywności
face_intensity = calculate_image_intensity(cv2.cvtColor(face_area, cv2.COLOR_RGB2GRAY))
context_intensity = calculate_image_intensity(cv2.cvtColor(context_area, cv2.COLOR_RGB2GRAY))
context_intensity_far = calculate_image_intensity(cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY))

print("Intensywność samej twarzy:", face_intensity)
print("Intensywność obszaru bliższego twarzy:", context_intensity)
print("Intensywność obszaru dalszego twarzy:", context_intensity_far)

# Logika porównania
# [PARAMETR]
threshold = 0.15

value = abs(context_intensity_far - context_intensity) / face_intensity
print("Wartość intensywności:", value)

if value > threshold:
    print("Wartość przekroczyła próg!")

plt.imshow(context_area_far)
plt.title("Dalszy kontekst obrazu")
plt.axis('off')
plt.show()

In [64]:
# Do pracy
# Testy - sprawdzenie intensywności dla różnych obrazów
import os

image_path = "../../../Dane/Sample/zd"

# Wczytanie obrazów
files = [f for f in os.listdir(image_path) if f.endswith(".jpg") or f.endswith(".png") or f.endswith(".webp")] # Tylko pliki JPG i PNG i WEBP

images = []
for file in files:
    img = cv2.imread(os.path.join(image_path, file), cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append(img)

# Przeprowadzenie analizy intensywności dla każdego obrazu
for img in images:
    faces = face_classifier.detectMultiScale(img, 1.3, 5)

    if faces is None or len(faces) == 0:
        continue

    face = faces[0]
    face_context = get_face_context(img, face)
    x, y, w, h = face

    # Bezpośrednie otoczenie twarzy
    rectangle_point = (int(face_context['rectangle_point'][0]), int(face_context['rectangle_point'][1]))
    rectangle_width = int(face_context['rectangle_width'])
    rectangle_height = int(face_context['rectangle_height'])

    # Dalsze otoczenie twarzy
    rectangle_point_far = (int(face_context['rectangle_point_far'][0]), int(face_context['rectangle_point_far'][1]))
    rectangle_width_far = int(face_context['rectangle_width_far'])
    rectangle_height_far = int(face_context['rectangle_height_far'])

    # Uzyskanie fragmentów obrazu
    face_area = img[y:y+h, x:x+w]
    context_area = img[rectangle_point[1]:rectangle_point[1] + rectangle_height, rectangle_point[0]:rectangle_point[0] + rectangle_width]
    context_area_far = img[rectangle_point_far[1]:rectangle_point_far[1] + rectangle_height_far, rectangle_point_far[0]:rectangle_point_far[0] + rectangle_width_far]

    # Obliczenie intensywności
    face_intensity = calculate_image_intensity(cv2.cvtColor(face_area, cv2.COLOR_RGB2GRAY))
    context_intensity = calculate_image_intensity(cv2.cvtColor(context_area, cv2.COLOR_RGB2GRAY))
    context_intensity_far = calculate_image_intensity(cv2.cvtColor(context_area_far, cv2.COLOR_RGB2GRAY))

    print("Intensywność samej twarzy:", face_intensity)
    print("Intensywność obszaru bliższego twarzy:", context_intensity)
    print("Intensywność obszaru dalszego twarzy:", context_intensity_far)

    # Logika porównania
    # [PARAMETR]
    threshold = 0.15

    value = abs(context_intensity_far - context_intensity) / face_intensity
    print("Wartość intensywności:", value)

    if value > threshold:
        print("Wartość przekroczyła próg!")